# Métricas de evaluación: regresión y clasificación

Notebook de repaso de las métricas que se calculan **con el conjunto de test** (o en la validación cruzada) para medir qué tan bien rinde un modelo con datos que no vio.

**Contenido**
1. Métricas de **regresión** (Y numérica: precio, nota, tiempo)
2. Métricas de **clasificación** (Y categórica: sí/no, moroso/cumplidor)
3. Cómo elegir la métrica
4. Tabla resumen

> **Recordá:** las métricas se calculan en el **test** (una sola vez, al final) o en la **validación** de la CV (para elegir modelos e hiperparámetros). Calculadas sobre el train, dan una imagen demasiado optimista.

**Funciones en R:** se indican del paquete `yardstick` (tidymodels) y alternativas de `Metrics`, `caret` y `pROC`.

---
# Parte 1: Métricas de regresión

**Ejemplo que usamos en toda esta parte:** un modelo predice el precio de 3 casas (en miles).

| Casa | Precio real ($y$) | Predicción ($\hat{y}$) | Error ($y - \hat{y}$) |
|---|---|---|---|
| A | 100 | 110 | −10 |
| B | 150 | 140 | +10 |
| C | 200 | 230 | −30 |

El **error** (o residuo) de cada casa es: real − predicho.

## 1.1 RSS: suma de los errores al cuadrado

$$RSS = \sum (y_i - \hat{y}_i)^2$$

**Qué es:** suma todos los errores, elevados al cuadrado.
**Para qué sirve:** es lo que **minimiza** la regresión lineal (OLS) al entrenar. Como métrica de evaluación se usa poco, porque crece con la cantidad de datos.

**Ejemplo:** $(-10)^2 + 10^2 + (-30)^2 = 100 + 100 + 900 =$ **1100**

**En R:** se calcula directo con `sum((real - pred)^2)`.

## 1.2 MSE: error cuadrático medio

$$MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2$$

**Qué es:** el **promedio** de los errores al cuadrado (el RSS dividido por la cantidad de datos).
**Cómo se lee:** "el error al cuadrado promedio es…". Está en **unidades al cuadrado** (miles²), así que no se interpreta directamente: sirve para **comparar** modelos.

**Ejemplo:** $1100 / 3 \approx$ **366,7** (miles²)

**En R:** `Metrics::mse()`

## 1.3 RMSE: raíz del error cuadrático medio

$$RMSE = \sqrt{MSE}$$

**Qué es:** la raíz del MSE. Vuelve a las **unidades originales** de Y.
**Cómo se lee:** "el modelo se equivoca **típicamente** por unos X". Le da **más peso a los errores grandes**, porque eleva al cuadrado antes de promediar.

**Ejemplo:** $\sqrt{366{,}7} \approx$ **19,1 mil**: el modelo se equivoca típicamente por unos 19 mil.

**En R:** `yardstick::rmse()` · `Metrics::rmse()` · `caret::RMSE()`

## 1.4 MAE: error absoluto medio

$$MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$$

**Qué es:** el **promedio** del tamaño de los errores, sin importar el signo (las barras $|\ |$ le quitan el signo).
**Cómo se lee:** "en promedio, el modelo se equivoca por X". Trata a **todos los errores por igual**, así que es **más robusto a los outliers** que el RMSE.

**Ejemplo:** $(10 + 10 + 30) / 3 \approx$ **16,7 mil**

**En R:** `yardstick::mae()` · `Metrics::mae()` · `caret::MAE()`

> **MAE contra RMSE:** el RMSE siempre da **igual o más** que el MAE. Si el RMSE es **bastante mayor**, hay **algunos errores grandes**. En el ejemplo, RMSE = 19,1 y MAE = 16,7: la casa C (error de 30) pesa más en el RMSE.

## 1.5 MAPE: error porcentual absoluto medio

$$MAPE = \frac{100\%}{n}\sum \left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

**Qué es:** el error promedio **en porcentaje** del valor real.
**Cómo se lee:** "en promedio, el modelo se equivoca en un X% del valor real". Es fácil de comunicar.
**Cuidado:** no sirve si hay valores reales iguales o cercanos a 0 (se divide por $y$).

**Ejemplo:** $10/100 = 10\%$; $10/150 = 6{,}7\%$; $30/200 = 15\%$. Promedio $\approx$ **10,6%**

**En R:** `yardstick::mape()` · `Metrics::mape()`

## 1.6 R²: coeficiente de determinación

$$R^2 = 1 - \frac{RSS}{\sum (y_i - \bar{y})^2}$$

**Qué es:** la **proporción de la variabilidad de Y que explica el modelo**, comparado con un modelo que siempre predice el promedio ($\bar{y}$).
**Cómo se lee:**
- $R^2 = 1$: el modelo explica todo (predicción perfecta).
- $R^2 = 0$: el modelo no es mejor que predecir siempre el promedio.
- En el test puede dar **negativo**, si el modelo es peor que el promedio.

**Ejemplo:** el promedio real es 150. La variabilidad total es $(100-150)^2 + (150-150)^2 + (200-150)^2 = 5000$.
$R^2 = 1 - 1100/5000 =$ **0,78** → el modelo explica el 78% de la variabilidad del precio.

**En R:** `yardstick::rsq()` · `caret::R2()` (en el train, `summary(lm)` muestra el R² y el R² ajustado)

> El **R² ajustado** penaliza agregar variables que no aportan. Se usa para comparar modelos con distinta cantidad de variables en el train.

---
# Parte 2: Métricas de clasificación

**Ejemplo que usamos en toda esta parte:** un modelo predice si un vecino será **moroso** (clase positiva = 1) o **cumplidor** (clase negativa = 0). En el test hay **100 vecinos**: 50 morosos y 50 cumplidores.

> **Primero definí cuál es la clase positiva:** es la que querés detectar (moroso, enfermo, fraude…). Todas las métricas dependen de esa elección.

## 2.1 Matriz de confusión

Cruza **lo real** con **lo que predijo el modelo** (después de aplicar un umbral a la probabilidad):

| | **Predijo: moroso (P)** | **Predijo: cumplidor (N)** |
|---|---|---|
| **Real: moroso** | **VP = 40** ✔ | **FN = 10** ✘ |
| **Real: cumplidor** | **FP = 5** ✘ | **VN = 45** ✔ |

- **VP (verdadero positivo):** era moroso y dijo moroso.
- **VN (verdadero negativo):** era cumplidor y dijo cumplidor.
- **FP (falso positivo):** era cumplidor y dijo moroso (**falsa alarma**).
- **FN (falso negativo):** era moroso y dijo cumplidor (**se le escapó**).

**Para recordar:** la **segunda letra** (P/N) es lo que **dijo el modelo**; la **primera** (V/F) dice si **acertó**.

**En R:** `yardstick::conf_mat()` · `caret::confusionMatrix()` (este último muestra también casi todas las métricas que siguen) · `table(real, predicho)`

## 2.2 Accuracy (exactitud)

$$Accuracy = \frac{VP + VN}{\text{total}}$$

**Qué es:** el **porcentaje de aciertos** sobre el total.
**Ejemplo:** $(40 + 45) / 100 =$ **0,85** → acierta en el 85% de los casos.

**Cuidado con las clases desbalanceadas:** si de 100 vecinos solo 5 son morosos, un modelo que dice **siempre "cumplidor"** tiene un accuracy de **95%**… ¡pero no detecta a ningún moroso! Por eso el accuracy solo, con clases desbalanceadas, engaña.

**En R:** `yardstick::accuracy()` · `Metrics::accuracy()`

## 2.3 Precision (precisión)

$$Precision = \frac{VP}{VP + FP}$$

**Qué responde:** "de los que el modelo **marcó como morosos**, ¿qué porcentaje **realmente lo era**?"
**Importa cuando las falsas alarmas son caras** (por ejemplo, iniciar un juicio por deuda a un cumplidor).

**Ejemplo:** $40 / (40 + 5) \approx$ **0,89** → el 89% de los marcados como morosos lo eran.

**En R:** `yardstick::precision()` · `Metrics::precision()`

## 2.4 Recall (sensibilidad, o tasa de verdaderos positivos, TPR)

$$Recall = \frac{VP}{VP + FN}$$

**Qué responde:** "de **todos los morosos reales**, ¿qué porcentaje **atrapó** el modelo?"
**Importa cuando que se escape un positivo es caro** (por ejemplo, no detectar una enfermedad).

**Ejemplo:** $40 / (40 + 10) =$ **0,80** → atrapó al 80% de los morosos (se le escaparon 10).

**En R:** `yardstick::recall()` · `yardstick::sens()` · `Metrics::recall()`

## 2.5 Specificity (especificidad) y FPR

$$Specificity = \frac{VN}{VN + FP} \qquad FPR = \frac{FP}{FP + VN} = 1 - Specificity$$

**Specificity:** "de **todos los cumplidores reales**, ¿qué porcentaje reconoció bien el modelo?"
**FPR (tasa de falsos positivos):** "de todos los cumplidores, ¿a qué porcentaje marcó mal como moroso?"

**Ejemplo:** Specificity $= 45 / 50 =$ **0,90** · FPR $= 5 / 50 =$ **0,10**

**En R:** `yardstick::spec()` · `caret::specificity()`

## 2.6 F1-score

$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

**Qué es:** un **promedio equilibrado** entre precision y recall (técnicamente, la media armónica). Es alto solo si **las dos** son altas.
**Para qué sirve:** resumir en un número cuando importan tanto las falsas alarmas como los casos que se escapan, sobre todo con **clases desbalanceadas**.

**Ejemplo:** $2 \cdot (0{,}89 \cdot 0{,}80) / (0{,}89 + 0{,}80) \approx$ **0,84**

**En R:** `yardstick::f_meas()` · `MLmetrics::F1_Score()`

## 2.7 Balanced accuracy (exactitud balanceada)

$$Balanced\ accuracy = \frac{Recall + Specificity}{2}$$

**Qué es:** el promedio de lo bien que le va al modelo **con cada clase por separado**. No se deja engañar por el desbalance.

**Ejemplo:** $(0{,}80 + 0{,}90) / 2 =$ **0,85**
En el caso del modelo que dice "siempre cumplidor" (con 5% de morosos): recall = 0 y specificity = 1, así que balanced accuracy = **0,50** (como el azar), aunque su accuracy sea 95%.

**En R:** `yardstick::bal_accuracy()`

## 2.8 Curva ROC

**Qué es:** un gráfico que muestra cómo le va al modelo **con todos los umbrales posibles**. Cada punto es un umbral:
- **Eje vertical:** TPR (recall). Más alto es mejor.
- **Eje horizontal:** FPR. Más bajo es mejor.

**Cómo se lee:**
- **La diagonal** representa un modelo que adivina al azar.
- **Cuanto más se pega la curva a la esquina de arriba a la izquierda**, mejor distingue el modelo entre clases.

**Para qué sirve:** evaluar y **comparar modelos sin depender de un umbral**, y ayudar a **elegir el umbral** según qué error cueste más.
**Se arma con:** las **probabilidades** (o puntajes) que da el modelo para el test, no con las clases ya asignadas.

**En R:** `pROC::roc()` + `plot()` · `yardstick::roc_curve()`

## 2.9 AUC: área bajo la curva ROC

**Qué es:** el área debajo de la curva ROC. Resume toda la curva en **un número entre 0 y 1**.

| AUC | Cómo se considera |
|---|---|
| 0,5 | Azar: no sirve |
| 0,6 a 0,7 | Pobre |
| 0,7 a 0,8 | Aceptable |
| 0,8 a 0,9 | Bueno |
| más de 0,9 | Muy bueno (si es casi 1, revisá que no haya un error en los datos) |

**Cómo se lee:** si tomás un moroso y un cumplidor al azar, el AUC es la **probabilidad de que el modelo le dé un puntaje más alto al moroso**. Ejemplo: AUC = 0,85 → ordena bien el par el 85% de las veces.

**En R:** `pROC::auc()` · `yardstick::roc_auc()` · `Metrics::auc()`

## 2.10 Log-loss (pérdida logarítmica)

$$LogLoss = -\frac{1}{n}\sum \left[y_i \log(p_i) + (1 - y_i)\log(1 - p_i)\right]$$

**Qué es:** evalúa **qué tan buenas son las probabilidades** que da el modelo (no las clases). **Más bajo es mejor.**
**La idea:** castiga más fuerte **estar muy seguro y equivocarse**.

**Ejemplo** (un vecino que **sí** es moroso, $y = 1$):
- El modelo dijo $p = 0{,}9$ → castigo $= -\log(0{,}9) \approx$ **0,11** (chico: acertó con seguridad).
- El modelo dijo $p = 0{,}1$ → castigo $= -\log(0{,}1) \approx$ **2,30** (grande: estaba seguro y se equivocó).

También es la **función de costo** que minimiza la regresión logística al entrenar.

**En R:** `yardstick::mn_log_loss()` · `Metrics::logLoss()` · `MLmetrics::LogLoss()`

---
# Parte 3: ¿Qué métrica uso?

**Regresión**
- Querés un error **fácil de explicar**, en las unidades de Y → **RMSE** o **MAE**.
- **Hay outliers** y no querés que dominen → **MAE**.
- **Los errores grandes son graves** y querés castigarlos → **RMSE**.
- Querés el error **en porcentaje** (y Y nunca es 0) → **MAPE**.
- Querés saber **cuánta variabilidad explica** el modelo → **R²**.

**Clasificación**
- Clases **balanceadas** y los dos errores cuestan parecido → **accuracy**.
- Clases **desbalanceadas** → **F1**, **balanced accuracy** o **AUC** (nunca el accuracy solo).
- Las **falsas alarmas** son caras → mirá la **precision**.
- Que **se escape un positivo** es caro → mirá el **recall**.
- Querés evaluar el modelo **sin depender del umbral**, o comparar modelos → **curva ROC y AUC**.
- Te importa que las **probabilidades** sean buenas (no solo la clase) → **log-loss**.

---
# Parte 4: Tabla resumen

| Tipo | Métrica | Fórmula (en palabras) | Mejor si es… | Función en R |
|---|---|---|---|---|
| Regresión | **RSS** | suma de errores² | más bajo | `sum((real - pred)^2)` |
| Regresión | **MSE** | promedio de errores² | más bajo | `Metrics::mse()` |
| Regresión | **RMSE** | raíz del MSE | más bajo | `yardstick::rmse()` |
| Regresión | **MAE** | promedio de errores sin signo | más bajo | `yardstick::mae()` |
| Regresión | **MAPE** | promedio del error en % | más bajo | `yardstick::mape()` |
| Regresión | **R²** | proporción de variabilidad explicada | más alto (máx. 1) | `yardstick::rsq()` |
| Clasificación | **Matriz de confusión** | cruce real × predicho | — | `caret::confusionMatrix()` |
| Clasificación | **Accuracy** | aciertos / total | más alto | `yardstick::accuracy()` |
| Clasificación | **Precision** | VP / (VP + FP) | más alto | `yardstick::precision()` |
| Clasificación | **Recall (TPR)** | VP / (VP + FN) | más alto | `yardstick::recall()` |
| Clasificación | **Specificity** | VN / (VN + FP) | más alto | `yardstick::spec()` |
| Clasificación | **F1** | equilibrio precision–recall | más alto | `yardstick::f_meas()` |
| Clasificación | **Balanced accuracy** | (recall + specificity) / 2 | más alto | `yardstick::bal_accuracy()` |
| Clasificación | **Curva ROC** | TPR vs. FPR en todos los umbrales | pegada arriba a la izquierda | `pROC::roc()` |
| Clasificación | **AUC** | área bajo la ROC | más alto (máx. 1) | `pROC::auc()` |
| Clasificación | **Log-loss** | castigo a probabilidades malas | más bajo | `yardstick::mn_log_loss()` |